# Harvest Commons mechanisms

This notebook inspects the new Harvest Commons batch after three implementation changes:

1. hybrid governance reacts to local aggressive neighborhoods
2. bottom-up credit signals affect later harvest behavior
3. local mechanism metrics are logged and summarized


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

TABLE = ROOT / "results/runs/harvest/curated/harvest_commons_batch8_neighbor_targeted_table.csv"
df = pd.read_csv(TABLE)
df


In [ ]:
condition_order = ["none", "top_down_only", "bottom_up_only", "hybrid"]
condition_labels = {
    "none": "None",
    "top_down_only": "Top-down only",
    "bottom_up_only": "Bottom-up only",
    "hybrid": "Hybrid",
}
condition_colors = {
    "none": "#8d99ae",
    "top_down_only": "#d1495b",
    "bottom_up_only": "#2a9d8f",
    "hybrid": "#264653",
}
social_mixes = df["social_mix"].drop_duplicates().tolist()

def plot_metric_grid(metrics, title):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
    axes = axes.flatten()
    width = 0.18
    x = np.arange(len(social_mixes))
    for ax, (metric, ylabel) in zip(axes, metrics):
        for idx, condition in enumerate(condition_order):
            subset = df[df["condition"] == condition].set_index("social_mix").reindex(social_mixes)
            ax.bar(x + (idx - 1.5) * width, subset[metric].to_numpy(), width=width, color=condition_colors[condition], label=condition_labels[condition])
        ax.set_xticks(x)
        ax.set_xticklabels([mix.replace("_", "\n") for mix in social_mixes])
        ax.set_ylabel(ylabel)
        ax.grid(axis="y", alpha=0.25)
    axes[0].legend(loc="upper left", ncols=2, fontsize=9)
    fig.suptitle(title, fontsize=16)
    return fig


In [ ]:
health_metrics = [
    ("mean_patch_health_mean", "Mean patch health"),
    ("mean_welfare_mean", "Mean welfare"),
    ("payoff_gini_mean", "Payoff Gini"),
    ("mean_patch_variance_mean", "Patch variance"),
]
plot_metric_grid(health_metrics, "Harvest Commons outcomes by social mix and governance")
plt.show()


In [ ]:
mechanism_metrics = [
    ("mean_aggressive_request_fraction_mean", "Aggressive request fraction"),
    ("mean_max_local_aggression_mean", "Max local aggression"),
    ("mean_capped_action_fraction_mean", "Capped action fraction"),
    ("mean_neighborhood_overharvest_mean", "Neighborhood overharvest"),
]
plot_metric_grid(mechanism_metrics, "Harvest Commons mechanisms by social mix and governance")
plt.show()


In [ ]:
comparison = []
for social_mix in social_mixes:
    top = df[(df["social_mix"] == social_mix) & (df["condition"] == "top_down_only")].iloc[0]
    hybrid = df[(df["social_mix"] == social_mix) & (df["condition"] == "hybrid")].iloc[0]
    comparison.append({
        "social_mix": social_mix,
        "hybrid_minus_top_patch_health": hybrid["mean_patch_health_mean"] - top["mean_patch_health_mean"],
        "hybrid_minus_top_welfare": hybrid["mean_welfare_mean"] - top["mean_welfare_mean"],
        "hybrid_minus_top_aggressive_fraction": hybrid["mean_aggressive_request_fraction_mean"] - top["mean_aggressive_request_fraction_mean"],
        "hybrid_minus_top_local_aggression": hybrid["mean_max_local_aggression_mean"] - top["mean_max_local_aggression_mean"],
        "hybrid_minus_top_neighborhood_overharvest": hybrid["mean_neighborhood_overharvest_mean"] - top["mean_neighborhood_overharvest_mean"],
    })
pd.DataFrame(comparison)


## What to look for

- In `cooperative_heavy`, top-down intervention should stay mostly inactive.
- In `mixed_pressure`, the tuned hybrid should nearly close the welfare gap against `top_down_only` while reducing local aggression and neighborhood overharvest.
- In `adversarial_heavy`, hybrid should still improve local mechanism metrics, but it may continue to trade some welfare for control.
